# 19 · Verify all uploaded PDFs are indexed

**Read-only audit** for your existing `group2-course-rag-engine` folder. Checks the *actual PDF bytes* (SHA-256), database status, presence of searchable chunks, and folder-to-domain mapping. Original seven-file tests alone do **not** prove nested folders are indexed.

Open this notebook in VS Code with the same `.venv` kernel and click **Run All**. A successful run shows **INDEXED** for every PDF. If a document says **NOT INDEXED**, the file exists but was never ingested into this project database. If it says **WRONG DOMAIN**, its classification needs correction. This check does not modify the database. The report is saved under `.runtime/pdf_index_audit.json`.

**Scope:** All `.pdf` files inside `data/course_sources/official_pdfs/` including subfolders. It does not extract PDFs that still live only inside a `.zip` archive.

In [1]:
from pathlib import Path
import sqlite3, hashlib, json
from collections import Counter, defaultdict

# Read-only audit: does NOT ingest files or alter your existing database.
current = Path.cwd().resolve()
root_candidates = [current, *current.parents]
ROOT = next((p for p in root_candidates
             if (p / 'notebooks' / '00_foundation.ipynb').is_file()
             and (p / 'requirements.txt').is_file()), None)
assert ROOT is not None, 'Open group2-course-rag-engine in VS Code and run this notebook from its notebooks folder.'
DOCS = ROOT / 'data' / 'course_sources' / 'official_pdfs'
DB = ROOT / '.runtime' / 'group2.sqlite3'
PROJECT_ID = '77777777-7777-4777-8777-777777777777'
assert DOCS.is_dir(), f'Cannot find your PDF folder: {DOCS}'
assert DB.is_file(), f'Cannot find indexed database: {DB}. Run the import notebooks first.'

# Expected domains are assigned from your existing folder structure.
domain_folders = {
    'Clinical': 'clinical',
    'FoodSecurity': 'csr',
    'UnitedStatesSecuritiesandExchangeCommission(SEC)': 'sec',
    'Technology': 'technology',
    'technology': 'technology',
    'clinical': 'clinical',
    'sec': 'sec',
    'csr': 'csr',
}
manifest_path = DOCS / 'official_document_manifest.json'
root_domains = {}
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    for row in manifest:
        root_domains[row['filename'].casefold()] = row['domain']

# Query the stored documents and their chunk counts, read-only.
cx = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
cx.row_factory = sqlite3.Row
try:
    stored = cx.execute("""
        SELECT d.id, d.filename, d.domain, d.sha256, d.status,
               COUNT(c.id) AS chunk_count
        FROM documents AS d
        LEFT JOIN chunks AS c ON c.document_id = d.id
        WHERE d.project_id = ?
        GROUP BY d.id
    """, (PROJECT_ID,)).fetchall()
    projects = cx.execute('SELECT DISTINCT project_id FROM documents').fetchall()
finally:
    cx.close()

by_hash = defaultdict(list)
for row in stored:
    by_hash[row['sha256']].append(dict(row))

pdfs = sorted(p for p in DOCS.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf')
report = []
for pdf in pdfs:
    rel = pdf.relative_to(DOCS)
    expected = domain_folders.get(rel.parts[0]) if len(rel.parts) > 1 else root_domains.get(pdf.name.casefold())
    with pdf.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest() if hasattr(hashlib,'file_digest') else hashlib.sha256(handle.read()).hexdigest()
    matches = by_hash.get(digest, [])
    valid = [m for m in matches if m['status'] == 'ready' and m['chunk_count'] > 0]
    same_domain = [m for m in valid if m['domain'] == expected] if expected else valid
    if same_domain:
        state = 'INDEXED' if expected else 'INDEXED — REVIEW DOMAIN'
    elif valid:
        state = 'WRONG DOMAIN'
    elif matches:
        state = 'NO SEARCHABLE CHUNKS'
    else:
        state = 'NOT INDEXED'
    report.append({'file': str(rel), 'expected': expected or 'UNKNOWN',
                   'state': state, 'indexed_domain': ', '.join(sorted({m['domain'] for m in matches})) or '-',
                   'chunks': max((m['chunk_count'] for m in matches), default=0), 'sha256': digest})

print('Project:', ROOT)
print('Database:', DB)
print('Project ID:', PROJECT_ID)
print('PDF files found:', len(pdfs), '| unique PDF contents:', len({x['sha256'] for x in report}))
print('Database documents for this project:', len(stored))
print('Stored project IDs:', ', '.join(x[0] for x in projects) or '(none)')
print('\nRESULTS BY STATUS:')
for key, n in sorted(Counter(x['state'] for x in report).items()):
    print(f'  {key}: {n}')
print('\nRESULTS BY EXPECTED DOMAIN:')
for key, n in sorted(Counter(x['expected'] for x in report).items()):
    print(f'  {key}: {n} file(s)')
problems = [x for x in report if x['state'] != 'INDEXED']
if problems:
    print('\nFILES NEEDING ATTENTION:')
    for r in problems:
        print(f"  {r['state']}: {r['file']} | expected={r['expected']} | indexed={r['indexed_domain']} | chunks={r['chunks']}")
else:
    print('\nPASS: Every PDF in official_pdfs has a ready document with searchable chunks in its expected domain.')

# This is a readable report, and can be copied into a Git handoff if desired.
result_file = ROOT / '.runtime' / 'pdf_index_audit.json'
result_file.write_text(json.dumps({'summary': dict(Counter(x['state'] for x in report)),
                                   'files': report}, indent=2))
print('\nFull report saved to:', result_file)


Project: /Users/nehlinshanila/Downloads/group2-course-rag-engine
Database: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Project ID: 77777777-7777-4777-8777-777777777777
PDF files found: 34 | unique PDF contents: 34
Database documents for this project: 34
Stored project IDs: 77777777-7777-4777-8777-777777777777

RESULTS BY STATUS:
  INDEXED: 34

RESULTS BY EXPECTED DOMAIN:
  clinical: 20 file(s)
  csr: 2 file(s)
  sec: 8 file(s)
  technology: 4 file(s)

PASS: Every PDF in official_pdfs has a ready document with searchable chunks in its expected domain.

Full report saved to: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/pdf_index_audit.json


## CSV data check

ClinicalTrials and Market Basket CSVs are *not PDF evidence*. They have independent ingestion workflows. The cell below only lists them so you can confirm the files are present.

In [2]:
from collections import Counter

# CSV files are DATASETS, not PDF evidence. Their inventory is checked separately.
course_dir = ROOT / 'data' / 'course_sources'
csves = sorted(p for p in course_dir.rglob('*') if p.is_file() and p.suffix.lower() == '.csv')
print('CSV datasets found:', len(csves))
for csv in csves:
    print(' -', csv.relative_to(ROOT), f'({csv.stat().st_size / 1024 / 1024:.1f} MB)')
print('\nNOTE: This audit verifies the PDF RAG index only. It does not confirm that the Retail CSV')
print('has been imported into the separate market-basket transaction database.')


CSV datasets found: 5
 - data/course_sources/official_pdfs/Clinical/ctg-studies_Hearattack.csv (11.7 MB)
 - data/course_sources/official_pdfs/Clinical/ctg-studies_KneeInjuries.csv (4.0 MB)
 - data/course_sources/official_pdfs/Clinical/ctg-studies_covid.csv (17.1 MB)
 - data/course_sources/official_pdfs/Clinical/ctg-studies_diabetes.csv (74.0 MB)
 - data/course_sources/official_pdfs/Retail 2/CMPE256_Hackathon_market_basket_analysis_Release.csv (0.2 MB)

NOTE: This audit verifies the PDF RAG index only. It does not confirm that the Retail CSV
has been imported into the separate market-basket transaction database.
